In [40]:
!pip install -U trl accelerate transformers bitsandbytes peft

In [41]:
!pip install PyMuPDF

In [42]:
from datasets import Dataset, load_dataset

**our costum data for domain specific fine tune**

In [43]:
import fitz

In [44]:
from google.colab import files
uploaded = files.upload()

Saving Rithik_Tiwari_Resume_Summary.pdf to Rithik_Tiwari_Resume_Summary (1).pdf


In [45]:
def extract_text_from_pdf(pdf_path):
  text_blocks= []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        text_blocks.append(text)
  return text_blocks

In [46]:
pdf_texts = extract_text_from_pdf("/content/Rithik_Tiwari_Resume_Summary.pdf")
print(f"Successfully extracted {len(pdf_texts)} text blocks from the PDF.")

Successfully extracted 2 text blocks from the PDF.


In [47]:
 pdf_texts

['Professional Profile & Career Overview\nRithik Tiwari | AI/ML Engineer\nProfessional Summary\nRithik Tiwari is an aspiring AI/ML Engineer specializing in production-ready machine learning systems, NLP pipelines,\nand intelligent multi-agent architectures. He has hands-on experience in developing scalable AI applications using\nPython, PyTorch, LangChain, Docker, and Airflow. His work demonstrates a strong focus on model optimization,\nautomation, MLOps workflows, and deploying efficient AI systems for real-world use cases.\nEducation\nChandigarh Engineering College, Mohali, Punjab\nBachelor of Technology (B.Tech) in Artificial Intelligence & Machine Learning (2022 – 2026)\nTechnical Skills\nCategory\nSkills\nProgramming\nPython, C++, R, SQL\nAI/ML\nMachine Learning, Deep Learning, NLP, Feature Engineering\nFrameworks\nPyTorch, TensorFlow, Scikit-learn, LangChain, Hugging Face\nMLOps\nMLflow, Airflow, Docker, Kafka, CI/CD Pipelines\nGenAI\nRAG Systems, Prompt Engineering, Multi-Agent 

In [48]:
import re

def split_paragraphs(pages):
    paragraphs = []

    for page_text in pages:

        # Split text wherever there are blank lines
        chunks = re.split(r'\n\s*\n', page_text)

        for chunk in chunks:
            clean = chunk.strip()

            # Ignore very small text pieces
            if len(clean) > 30:
                paragraphs.append(clean)

    return paragraphs

In [49]:
paragraphs = split_paragraphs(pdf_texts)
paragraphs

['Professional Profile & Career Overview\nRithik Tiwari | AI/ML Engineer\nProfessional Summary\nRithik Tiwari is an aspiring AI/ML Engineer specializing in production-ready machine learning systems, NLP pipelines,\nand intelligent multi-agent architectures. He has hands-on experience in developing scalable AI applications using\nPython, PyTorch, LangChain, Docker, and Airflow. His work demonstrates a strong focus on model optimization,\nautomation, MLOps workflows, and deploying efficient AI systems for real-world use cases.\nEducation\nChandigarh Engineering College, Mohali, Punjab\nBachelor of Technology (B.Tech) in Artificial Intelligence & Machine Learning (2022 – 2026)\nTechnical Skills\nCategory\nSkills\nProgramming\nPython, C++, R, SQL\nAI/ML\nMachine Learning, Deep Learning, NLP, Feature Engineering\nFrameworks\nPyTorch, TensorFlow, Scikit-learn, LangChain, Hugging Face\nMLOps\nMLflow, Airflow, Docker, Kafka, CI/CD Pipelines\nGenAI\nRAG Systems, Prompt Engineering, Multi-Agent 

In [50]:
data = [{"text": p}for p in paragraphs]
data

[{'text': 'Professional Profile & Career Overview\nRithik Tiwari | AI/ML Engineer\nProfessional Summary\nRithik Tiwari is an aspiring AI/ML Engineer specializing in production-ready machine learning systems, NLP pipelines,\nand intelligent multi-agent architectures. He has hands-on experience in developing scalable AI applications using\nPython, PyTorch, LangChain, Docker, and Airflow. His work demonstrates a strong focus on model optimization,\nautomation, MLOps workflows, and deploying efficient AI systems for real-world use cases.\nEducation\nChandigarh Engineering College, Mohali, Punjab\nBachelor of Technology (B.Tech) in Artificial Intelligence & Machine Learning (2022 – 2026)\nTechnical Skills\nCategory\nSkills\nProgramming\nPython, C++, R, SQL\nAI/ML\nMachine Learning, Deep Learning, NLP, Feature Engineering\nFrameworks\nPyTorch, TensorFlow, Scikit-learn, LangChain, Hugging Face\nMLOps\nMLflow, Airflow, Docker, Kafka, CI/CD Pipelines\nGenAI\nRAG Systems, Prompt Engineering, Mul

In [51]:
dataset = Dataset.from_list(data)


In [52]:
dataset

Dataset({
    features: ['text'],
    num_rows: 2
})

**LETS SELECT THE MODEL**

In [53]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [54]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [55]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [56]:
def tokenizer_fn(examples):
  tokens= tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
  tokens["labels"]=  tokens["input_ids"].copy()
  return tokens

In [57]:
tokenized = dataset.map(tokenizer_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [58]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})

In [59]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [60]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps =500,
    save_total_limit= 2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to='None'
)

In [ ]:
from transformers import TrainingArguments
help(TrainingArguments)

# **now let use LORA method**

In [61]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install -U trl accelerate transformers bitsandbytes peft

In [64]:
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset


In [65]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [66]:
tokenizer = AutoTokenizer.from_pretrained(model)


In [67]:
if tokenizer.pad_token is None:
  tokenizer.pad_token= tokenizer.eos_token

In [68]:
def tokenize_fn(examples):
  tokens = tokenizer(
      examples["text"],
      truncation=True,
      padding="max_length",
      max_length=512
  )
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens


In [69]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [71]:
tokenized


Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})

In [73]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [75]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",

)

In [76]:
q_lora_model =  get_peft_model(model, lora_config)

In [88]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to=[]
)

In [89]:
trainer = Trainer(
    model=q_lora_model,
    args=args,
    train_dataset=tokenized

)

In [90]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=5, training_loss=6.792807006835938, metrics={'train_runtime': 10.7827, 'train_samples_per_second': 0.927, 'train_steps_per_second': 0.464, 'total_flos': 31814823444480.0, 'train_loss': 6.792807006835938, 'epoch': 5.0})

In [92]:
model_path = "/content/tinyllama-lora/checkpoint-5"

In [94]:
!pip install --upgrade torchao
Trained_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [96]:
prompt = "who is Rithik Tiwari"

In [100]:
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {name: tensor.to(Trained_model.device) for name, tensor in inputs.items()}

In [102]:
import torch

output = Trained_model.generate(
    **{k: v.to(Trained_model.device) for k, v in inputs.items()},
    max_new_tokens=100,
    temperature=0.8,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [103]:
decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
print(decoded_output)

who is Rithik Tiwari, has been given the role of the lead male protagonist in this movie.
Most of the actors have already signed the film and they are working hard to complete the shooting schedule before the release of their upcoming movie. The movie is going to be released on July 10, 2019. The makers are planning to promote the movie in India in the last quarter of the year.
Due to the upcoming release of their upcoming movie, the film
